In [87]:
!pip install -q chromadb sentence-transformers

In [88]:
import chromadb

In [89]:
from sentence_transformers import SentenceTransformer

In [90]:
client=chromadb.PersistentClient(path="./chroma_db")

In [91]:
collection=client.get_or_create_collection(name="SemanticSearch")

In [92]:
model=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [93]:
documents = [
    "I love learning artificial intelligence.",
    "Machine learning is an interesting field.",
    "Deep learning is a subset of machine learning.",
    "Python is widely used for AI development.",
    "I enjoy playing cricket.",
    "Neural networks are inspired by the human brain.",
    "Vector databases store numerical embeddings.",
    "ChromaDB is a vector database.",
    "I like watching football.",
    "Natural language processing helps computers understand text."
]

print("Total documents:", len(documents))

Total documents: 10


In [94]:
embeddings=model.encode(documents)

In [95]:
embeddings.shape

(10, 384)

In [96]:
ids=[f"doc_{i}" for i in range(len(documents))]

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist()
    )
print("Documents inserted successfully!")
print("Total records:", collection.count())

Documents inserted successfully!
Total records: 10


In [97]:
stored_data=collection.get(include=["documents","embeddings"])
print("IDs : ",stored_data["ids"])
print("\ndocuments : ")
for doc in stored_data["documents"]:
    print("-", doc)
print("embeddings : ",stored_data["embeddings"].shape)


IDs :  ['doc_0', 'doc_1', 'doc_2', 'doc_3', 'doc_4', 'doc_5', 'doc_6', 'doc_7', 'doc_8', 'doc_9']

documents : 
- I love learning artificial intelligence.
- Machine learning is an interesting field.
- Deep learning is a subset of machine learning.
- Python is widely used for AI development.
- I enjoy playing cricket.
- Neural networks are inspired by the human brain.
- Vector databases store numerical embeddings.
- ChromaDB is a vector database.
- I like watching football.
- Natural language processing helps computers understand text.
embeddings :  (10, 384)


In [98]:
type(stored_data)

dict

In [99]:
query = "I want to learn about AI and machine learning"

query_embedding=model.encode(query)
results=collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3
)
print("search_query : ",query)
print("\nTop 3 Results : ")
for i, doc in enumerate(results["documents"][0]):
    print("S.no : ",i+1,doc)
    print("ID:", results["ids"][0][i])
    print("Distance:", results["distances"][0][i])


search_query :  I want to learn about AI and machine learning

Top 3 Results : 
S.no :  1 I love learning artificial intelligence.
ID: doc_0
Distance: 0.7630816102027893
S.no :  2 Python is widely used for AI development.
ID: doc_3
Distance: 0.8844349384307861
S.no :  3 Machine learning is an interesting field.
ID: doc_1
Distance: 0.9175742268562317


In [100]:
def SemanticSearch(query,k):
  query_embedding=model.encode(query)

  results=collection.query(query_embeddings=query_embedding.tolist(),n_results=k)
  print("Searched : ",query)
  print("\nTop ",k," Results : ")
  for  i, doc in enumerate(results["documents"][0]):
    print("S.no : ",i+1,doc)
    print("ID:", results["ids"][0][i])
    print("Distance:", results["distances"][0][i])

In [101]:
search=input("Search your Question ")
s=SemanticSearch(search,3)


Search your Question who are you?
Searched :  who are you?

Top  3  Results : 
S.no :  1 I like watching football.
ID: doc_8
Distance: 1.6747190952301025
S.no :  2 I enjoy playing cricket.
ID: doc_4
Distance: 1.6828666925430298
S.no :  3 I love learning artificial intelligence.
ID: doc_0
Distance: 1.7442830801010132


In [102]:
metadata = [
    {"category": "AI"},
    {"category": "AI"},
    {"category": "AI"},
    {"category": "AI"},
    {"category": "Sports"},
    {"category": "AI"},
    {"category": "VectorDB"},
    {"category": "VectorDB"},
    {"category": "Sports"},
    {"category": "NLP"}
]

In [103]:
collection.update(
    ids=ids,
    metadatas=metadata
)

In [104]:
data = collection.get(
    include=["documents", "metadatas"]
)

for i in range(len(data["ids"])):
    print("ID:", data["ids"][i])
    print("Document:", data["documents"][i])
    print("Metadata:", data["metadatas"][i])
    print("-" * 50)

ID: doc_0
Document: I love learning artificial intelligence.
Metadata: {'category': 'AI', 'difficulty': 'Beginner'}
--------------------------------------------------
ID: doc_1
Document: Machine learning is an interesting field.
Metadata: {'difficulty': 'Beginner', 'category': 'AI'}
--------------------------------------------------
ID: doc_2
Document: Deep learning is a subset of machine learning.
Metadata: {'difficulty': 'Intermediate', 'category': 'AI'}
--------------------------------------------------
ID: doc_3
Document: Python is widely used for AI development.
Metadata: {'difficulty': 'Beginner', 'category': 'AI'}
--------------------------------------------------
ID: doc_4
Document: I enjoy playing cricket.
Metadata: {'difficulty': 'Beginner', 'category': 'Sports'}
--------------------------------------------------
ID: doc_5
Document: Neural networks are inspired by the human brain.
Metadata: {'category': 'AI', 'difficulty': 'Intermediate'}
-------------------------------------

In [105]:
metadata = [
    {"category": "AI", "difficulty": "Beginner"},
    {"category": "AI", "difficulty": "Beginner"},
    {"category": "AI", "difficulty": "Intermediate"},
    {"category": "AI", "difficulty": "Beginner"},
    {"category": "Sports", "difficulty": "Beginner"},
    {"category": "AI", "difficulty": "Intermediate"},
    {"category": "VectorDB", "difficulty": "Intermediate"},
    {"category": "VectorDB", "difficulty": "Beginner"},
    {"category": "Sports", "difficulty": "Beginner"},
    {"category": "NLP", "difficulty": "Intermediate"}
]

collection.update(
    ids=ids,
    metadatas=metadata
)

print("Metadata updated!")

Metadata updated!


In [106]:
query = "I want to learn about computers and intelligence"

query_embedding = model.encode(query)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    where={
        "$and": [
            {"category": "AI"},
            {"difficulty": "Beginner"}
        ]
    }
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}: {doc}")
    print("Category:", results["metadatas"][0][i]["category"])
    print("Difficulty: ",results["metadatas"][0][i]["difficulty"])


Result 1: I love learning artificial intelligence.
Category: AI
Difficulty:  Beginner

Result 2: Python is widely used for AI development.
Category: AI
Difficulty:  Beginner

Result 3: Machine learning is an interesting field.
Category: AI
Difficulty:  Beginner


In [107]:
import chromadb

print("ChromaDB version:", chromadb.__version__)

print("\nCollection name:", collection.name)

print("\nCollection configuration:")
print(collection.configuration)

print("\nCollection metadata:")
print(collection.metadata)

ChromaDB version: 1.5.9

Collection name: SemanticSearch

Collection configuration:
{'hnsw': {'space': 'l2', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': <chromadb.api.types.DefaultEmbeddingFunction object at 0x78f9e0598870>}

Collection metadata:
None


In [108]:
large_documents = [
    documents[i % len(documents)] + f" Example number {i}"
    for i in range(1000)
]

large_ids = [
    f"large_doc_{i}" for i in range(1000)
]

print("Total documents:", len(large_documents))

Total documents: 1000


In [109]:
large_embeddings = model.encode(
    large_documents,
    batch_size=64,
    show_progress_bar=True
)

print("Embedding shape:", large_embeddings.shape)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding shape: (1000, 384)


In [110]:
large_collection = client.get_or_create_collection(
    name="large_learning_material"
)

large_collection.add(
    ids=large_ids,
    documents=large_documents,
    embeddings=large_embeddings.tolist()
)

print("Total records:", large_collection.count())

Total records: 1000


In [111]:
large_collection.configuration

{'hnsw': {'space': 'l2',
  'ef_construction': 100,
  'ef_search': 100,
  'max_neighbors': 16,
  'resize_factor': 1.2,
  'sync_threshold': 1000},
 'spann': None,
 'embedding_function': <chromadb.api.types.DefaultEmbeddingFunction at 0x78f9e037b9d0>}

In [112]:
large_collection.metadata

In [113]:
query="i want to learn AI"

lquery_embedding=model.encode(query)
results=large_collection.query(
    query_embeddings=lquery_embedding.tolist(),
    n_results=3
)
for i in range(len(results["documents"][0])):
  print(f"\nResult {i+1}: {results['documents'][0][i]}")


Result 1: I love learning artificial intelligence. Example number 250

Result 2: I love learning artificial intelligence. Example number 350

Result 3: I love learning artificial intelligence. Example number 400


In [115]:
hnsw_collection = client.get_or_create_collection(
    name="hnsw_cosine_experiment",
    metadata={"hnsw:space": "cosine"}
)

hnsw_collection.add(
    ids=large_ids,
    documents=large_documents,
    embeddings=large_embeddings.tolist()
)

print("HNSW experiment collection created!")
print("Total records:", hnsw_collection.count())

HNSW experiment collection created!
Total records: 1000


In [116]:
query = "I want to learn about artificial intelligence"

query_embedding = model.encode(query)

results = hnsw_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}: {doc}")
    print("Distance:", results["distances"][0][i])


Result 1: I love learning artificial intelligence. Example number 250
Distance: 0.2777355909347534

Result 2: I love learning artificial intelligence. Example number 400
Distance: 0.28211188316345215

Result 3: I love learning artificial intelligence. Example number 670
Distance: 0.2823871374130249

Result 4: I love learning artificial intelligence. Example number 200
Distance: 0.28360801935195923

Result 5: I love learning artificial intelligence. Example number 940
Distance: 0.284989595413208


In [117]:
query = "I want to learn about artificial intelligence"

query_embedding = model.encode(query).tolist()

# Search using L2 distance
l2_results = large_collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

# Search using Cosine distance
cosine_results = hnsw_collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

print("========== L2 RESULTS ==========")

for i, doc in enumerate(l2_results["documents"][0]):
    print(f"\n{i+1}. {doc}")
    print("Distance:", l2_results["distances"][0][i])

print("\n========== COSINE RESULTS ==========")

for i, doc in enumerate(cosine_results["documents"][0]):
    print(f"\n{i+1}. {doc}")
    print("Distance:", cosine_results["distances"][0][i])

========== L2 RESULTS ==========

1. I love learning artificial intelligence. Example number 250
Distance: 0.5554712414741516

2. I love learning artificial intelligence. Example number 400
Distance: 0.5642238855361938

3. I love learning artificial intelligence. Example number 670
Distance: 0.5647743940353394

4. I love learning artificial intelligence. Example number 200
Distance: 0.5672162175178528

5. I love learning artificial intelligence. Example number 940
Distance: 0.5699788928031921

========== COSINE RESULTS ==========

1. I love learning artificial intelligence. Example number 250
Distance: 0.2777355909347534

2. I love learning artificial intelligence. Example number 400
Distance: 0.28211188316345215

3. I love learning artificial intelligence. Example number 670
Distance: 0.2823871374130249

4. I love learning artificial intelligence. Example number 200
Distance: 0.28360801935195923

5. I love learning artificial intelligence. Example number 940
Distance: 0.28498959541320

In [118]:
import numpy as np

for i in range(5):
    norm = np.linalg.norm(large_embeddings[i])
    print(f"Embedding {i}: Norm = {norm:.6f}")

Embedding 0: Norm = 1.000000
Embedding 1: Norm = 1.000000
Embedding 2: Norm = 1.000000
Embedding 3: Norm = 1.000000
Embedding 4: Norm = 1.000000


In [119]:
print("HNSW configuration:")
print(hnsw_collection.configuration["hnsw"])

HNSW configuration:
{'space': 'cosine', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}


In [120]:
import time

query = "I want to learn about artificial intelligence"
query_embedding = model.encode(query).tolist()

start = time.perf_counter()

results = hnsw_collection.query(
    query_embeddings=[query_embedding],
    n_results=10
)

end = time.perf_counter()

print("Search time:", round((end - start) * 1000, 3), "ms")
print("Results returned:", len(results["ids"][0]))

Search time: 5.191 ms
Results returned: 10


In [121]:
print(
    "Current ef_search:",
    hnsw_collection.configuration["hnsw"]["ef_search"]
)

Current ef_search: 100


In [125]:
low_ef_collection = client.get_or_create_collection(
    name="hnsw_ef10_experiment",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_search": 10
        }
    }
)

low_ef_collection.add(
    ids=large_ids,
    documents=large_documents,
    embeddings=large_embeddings.tolist()
)

print("Collection created!")
print("Configuration:", low_ef_collection.configuration["hnsw"])
print("Records:", low_ef_collection.count())

Collection created!
Configuration: {'space': 'cosine', 'ef_construction': 100, 'ef_search': 10, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}
Records: 1000


In [126]:
high_ef_collection = client.create_collection(
    name="hnsw_ef200_experiment",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_search": 200
        }
    }
)

high_ef_collection.add(
    ids=large_ids,
    documents=large_documents,
    embeddings=large_embeddings.tolist()
)

print("Collection created!")
print("Configuration:", high_ef_collection.configuration["hnsw"])
print("Records:", high_ef_collection.count())

Collection created!
Configuration: {'space': 'cosine', 'ef_construction': 100, 'ef_search': 200, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}
Records: 1000


In [127]:
import time

query = "I want to learn about artificial intelligence"
query_embedding = model.encode(query).tolist()

collections = {
    "ef_search = 10": low_ef_collection,
    "ef_search = 100": hnsw_collection,
    "ef_search = 200": high_ef_collection
}

benchmark_results = {}

for name, col in collections.items():

    # Warm-up query
    col.query(
        query_embeddings=[query_embedding],
        n_results=10
    )

    times = []

    # Run search 10 times
    for _ in range(10):
        start = time.perf_counter()

        result = col.query(
            query_embeddings=[query_embedding],
            n_results=10
        )

        end = time.perf_counter()
        times.append((end - start) * 1000)

    benchmark_results[name] = {
        "average_time": sum(times) / len(times),
        "ids": result["ids"][0]
    }

    print(f"\n{name}")
    print("Average search time:", round(sum(times) / len(times), 3), "ms")
    print("Results returned:", len(result["ids"][0]))


ef_search = 10
Average search time: 1.396 ms
Results returned: 10

ef_search = 100
Average search time: 1.308 ms
Results returned: 10

ef_search = 200
Average search time: 1.583 ms
Results returned: 10


In [128]:
reference_ids = set(
    benchmark_results["ef_search = 200"]["ids"]
)

for name, data in benchmark_results.items():

    current_ids = set(data["ids"])

    overlap = len(current_ids.intersection(reference_ids))

    print(f"\n{name}")
    print("Matching results:", overlap, "/ 10")


ef_search = 10
Matching results: 10 / 10

ef_search = 100
Matching results: 10 / 10

ef_search = 200
Matching results: 10 / 10


In [129]:
import time

query = "I want to learn about artificial intelligence"
query_embedding = model.encode(query).tolist()

collections = {
    "ef_search = 10": low_ef_collection,
    "ef_search = 100": hnsw_collection,
    "ef_search = 200": high_ef_collection
}

benchmark_results = {}

for name, col in collections.items():

    # Warm-up query
    col.query(
        query_embeddings=[query_embedding],
        n_results=10
    )

    times = []

    # Run search 10 times
    for _ in range(10):
        start = time.perf_counter()

        result = col.query(
            query_embeddings=[query_embedding],
            n_results=10
        )

        end = time.perf_counter()
        times.append((end - start) * 1000)

    benchmark_results[name] = {
        "average_time": sum(times) / len(times),
        "ids": result["ids"][0]
    }

    print(f"\n{name}")
    print("Average search time:", round(sum(times) / len(times), 3), "ms")
    print("Results returned:", len(result["ids"][0]))


ef_search = 10
Average search time: 1.106 ms
Results returned: 10

ef_search = 100
Average search time: 1.229 ms
Results returned: 10

ef_search = 200
Average search time: 1.249 ms
Results returned: 10


In [130]:
reference_ids = set(
    benchmark_results["ef_search = 200"]["ids"]
)

for name, data in benchmark_results.items():

    current_ids = set(data["ids"])

    overlap = len(current_ids.intersection(reference_ids))

    print(f"\n{name}")
    print("Matching results:", overlap, "/ 10")


ef_search = 10
Matching results: 10 / 10

ef_search = 100
Matching results: 10 / 10

ef_search = 200
Matching results: 10 / 10


In [132]:
reference_ids = set(
    benchmark_results["ef_search = 200"]["ids"]
)

for name, data in benchmark_results.items():

    current_ids = set(data["ids"])

    overlap = len(current_ids.intersection(reference_ids))

    print(f"\n{name}")
    print("Matching results:", overlap, "/ 10")


ef_search = 10
Matching results: 10 / 10

ef_search = 100
Matching results: 10 / 10

ef_search = 200
Matching results: 10 / 10
